In [0]:
# Gold Layer Quality Test Script
# English comments: Validating Star Schema integrity and referential checks

from pyspark.sql.functions import col, count

# 1. Load Gold Tables
df_fact = spark.table("workspace.gold.fact_trips")
df_dim_vendor = spark.table("workspace.gold.dim_vendor")
df_dim_payment = spark.table("workspace.gold.dim_payment_type")
df_dim_rate = spark.table("workspace.gold.dim_rate_code")
df_dim_date = spark.table("workspace.gold.dim_date")

# --- Test 1: Uniqueness of Surrogate Keys in Dimensions ---
# Each SK in a dimension must be unique (No duplicates)
def check_unique_sk(df, sk_column, table_name):
    duplicate_count = df.groupBy(sk_column).count().filter(col("count") > 1).count()
    if duplicate_count == 0:
        print(f"PASSED - {table_name}: All Surrogate Keys are unique.")
    else:
        raise Exception(f"FAILED - {table_name}: Found {duplicate_count} duplicate Surrogate Keys!")

check_unique_sk(df_dim_vendor, "vendor_sk", "dim_vendor")
check_unique_sk(df_dim_payment, "payment_type_sk", "dim_payment_type")
check_unique_sk(df_dim_rate, "rate_code_sk", "dim_rate_code")
check_unique_sk(df_dim_date, "date_key", "dim_date")

# --- Test 2: Referential Integrity (Foreign Key Check) ---
# Ensure every SK in Fact table exists in its respective Dim table
def check_referential_integrity(fact_df, dim_df, sk_column, label):
    invalid_links = fact_df.join(dim_df, sk_column, "left_anti").count()
    if invalid_links == 0:
        print(f"PASSED - Referential Integrity: {label} links are correct.")
    else:
        raise Exception(f"FAILED - Referential Integrity: {invalid_links} rows in Fact have no matching {label}!")

check_referential_integrity(df_fact, df_dim_vendor, "vendor_sk", "Vendor")
check_referential_integrity(df_fact, df_dim_payment, "payment_type_sk", "Payment Type")
check_referential_integrity(df_fact, df_dim_rate, "rate_code_sk", "Rate Code")
check_referential_integrity(df_fact, df_dim_date, "date_key", "Date")

# --- Test 3: Null Check in Fact Table Keys ---
# No SK or Date Key should be NULL in the Fact Table
null_keys = df_fact.filter(
    col("trip_sk").isNull() | 
    col("vendor_sk").isNull() | 
    col("payment_type_sk").isNull() |
    col("rate_code_sk").isNull() |
    col("date_key").isNull()
).count()

if null_keys == 0:
    print("PASSED - Null Check: No null keys found in Fact table.")
else:
    raise Exception(f"FAILED - Null Check: Found {null_keys} rows with null keys!")

# --- Test 4: Row Count Match ---
# Ensure we didn't lose any of our rows during the Joins
silver_count = spark.table("workspace.silver.cleaned_yellow_taxi").count()
gold_count = df_fact.count()

if gold_count == silver_count:
    print(f"PASSED - Row Count Check: Gold layer kept all {gold_count} rows.")
else:
    print(f"WARNING - Row count mismatch! Silver: {silver_count}, Gold: {gold_count}")